In [1]:
%load_ext autoreload
%autoreload 2

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import utils

# Importiamo i nostri moduli rifattorizzati!
from dataset import load_raw_data, encode_labels, get_holdout_dataloaders, create_dataloader
from models import MLP
from losses import PolyLoss, CombinedCLoss
from engine import train_model, evaluate_model
from plot_utils import plot_training_history
from gdv_utils import calculate_gdv

# Seed e Device
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo in uso: {device}\n")

# 1. Caricamento e preparazione dati
X_raw, y_raw_text = load_raw_data()
y_raw, num_classes = encode_labels(y_raw_text)

# 2. Creazione Dataloader per gli esperimenti Hold-Out
train_loader, val_loader, test_loader = get_holdout_dataloaders(X_raw, y_raw, batch_size=64)

Dispositivo in uso: cuda

Scaricamento del dataset Letter Recognition (UCI ID 59)...


In [2]:
import torch
import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
import time

# Assumiamo che calculate_gdv sia importabile dal tuo modulo
from gdv_utils import calculate_gdv 

# 1. MAPPING DEI DATASET COMPLETO (Nome -> UCI ID)
UCI_DATASETS = {
	"Autism Screening Adult": 426,
	"Bank Marketing": 222,
	"Blood Transfusion": 176,
	"Breast Cancer": 14,
	"Breast Cancer Coimbra": 451,
	"Breast Cancer Wisconsin (Diag.)": 17,
	"Cardiotocography": 193,
	"Chess (King-Rook vs. Pawn)": 22,
	"Cirrhosis Patient Survival": 878,
	"Congressional Voting": 105,
	"Contraceptive Method Choice": 30,
	"Credit Approval": 27,
	"Diff. Thyroid Cancer Recurrence": 848,
	"Drug Consumption": 373,
	"Haberman's Survival": 43,
	"Hayes-Roth": 44,
	"Heart Disease": 45,
	"Hepatitis C Virus (HCV)": 571,
	"ILPD (Indian Liver Patient)": 225,
	"ISOLET": 54,
	"Image Segmentation": 50,
	"Ionosphere": 52,
	"Letter Recognition": 59,
	"Mammographic Mass": 161,
	"Maternal Health Risk": 862,
	"Molecular Biology (Splice)": 69,
	"Musk (Version 2)": 75,
	"National Poll on Healthy Aging": 936,
	"NHANES Age Prediction": 887,
	"Optical / Pen-Based Digits": 80,
	"Page Blocks Classification": 78,
	"Polish Companies Bankruptcy": 365,
	"Predict Students' Dropout": 697,
	"Raisin": 850,
	"Room Occupancy Estimation": 864,
	"Spambase": 94,
	"SPECTF Heart": 96,
	"Statlog (Australian)": 143,
	"Statlog (German)": 144,
	"Statlog (Vehicle Silhouettes)": 149,
	"Steel Plates Faults": 198,
	"Student Performance": 320,
	"Taiwanese Bankruptcy": 572,
	"Vertebral Column": 212,
	"Waveform Database Gen.": 107,
	"Website Phishing": 379,
	"Yeast": 110
}

def clean_and_prepare_data(X, y):
	"""
	Pipeline di pulizia universale per i dataset UCI.
	Risolve NaN, feature testuali e ridimensionamenti.
	"""
	if not isinstance(X, pd.DataFrame):
		X = pd.DataFrame(X)
	if not isinstance(y, (pd.DataFrame, pd.Series)):
		y = pd.Series(y.ravel() if hasattr(y, 'ravel') else y)
		
	valid_idx = y.dropna().index
	X = X.loc[valid_idx].copy()
	y = y.loc[valid_idx].copy()

	X = pd.get_dummies(X, drop_first=True)

	imputer = SimpleImputer(strategy='mean')
	X_imputed = imputer.fit_transform(X)

	scaler = StandardScaler()
	X_scaled = scaler.fit_transform(X_imputed)

	encoder = LabelEncoder()
	y_encoded = encoder.fit_transform(y.values.ravel())

	X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
	y_tensor = torch.tensor(y_encoded, dtype=torch.long)

	return X_tensor, y_tensor, len(encoder.classes_)

def run_mass_gdv_analysis():
	print(f"Inizio analisi GDV su {len(UCI_DATASETS)} dataset...\n")
	
	risultati = []
	
	for nome, uci_id in UCI_DATASETS.items():
		print(f"Elaborazione: {nome} (ID: {uci_id})...", end=" ")
		start_time = time.time()
		
		try:
			dataset = fetch_ucirepo(id=uci_id)
			X_raw = dataset.data.features
			y_raw = dataset.data.targets
			
			if y_raw is None or y_raw.empty:
				raise ValueError("Nessun target (y) trovato nel dataset.")
				
			X_tensor, y_tensor, num_classes = clean_and_prepare_data(X_raw, y_raw)
			
			gdv = calculate_gdv(X_tensor, y_tensor)
			
			tempo = time.time() - start_time
			print(f"Completato! GDV: {gdv:.4f} | Tempo: {tempo:.2f}s")
			
			risultati.append({
				"Dataset": nome,
				"UCI_ID": uci_id,
				"Num_Campioni": X_tensor.shape[0],
				"Num_Feature_Post_Clean": X_tensor.shape[1],
				"Num_Classi": num_classes,
				"GDV_Spazio_Input": round(gdv, 4),
				"Status": "OK",
				"Note_Errore": "" # Nessun errore
			})
			
		except Exception as e:
			# Catturiamo il tipo di errore e il messaggio testuale
			messaggio_errore = f"{type(e).__name__}: {str(e)}"
			print(f"FALLITO! Motivo: {messaggio_errore}")
			
			risultati.append({
				"Dataset": nome,
				"UCI_ID": uci_id,
				"Num_Campioni": None,
				"Num_Feature_Post_Clean": None,
				"Num_Classi": None,
				"GDV_Spazio_Input": None,
				"Status": "Fallito",
				"Note_Errore": messaggio_errore # Salviamo il motivo dettagliato
			})
			
	df_report = pd.DataFrame(risultati)
	
	csv_filename = "report_gdv_uci_dettagliato.csv"
	df_report.to_csv(csv_filename, index=False)
	
	print("\n" + "="*60)
	print("ANALISI COMPLETATA!")
	print(f"I risultati sono stati salvati nel file: '{csv_filename}'")
	print("="*60)
	
	return df_report

if __name__ == "__main__":
	df_finale = run_mass_gdv_analysis()
	
	print("\n--- RIEPILOGO ESECUZIONE ---")
	totali = len(df_finale)
	successi = len(df_finale[df_finale['Status'] == 'OK'])
	fallimenti = totali - successi
	print(f"Dataset analizzati con successo: {successi}/{totali}")
	print(f"Dataset falliti: {fallimenti}/{totali}")
	
	if fallimenti > 0:
		print("\n--- DETTAGLIO FALLIMENTI ---")
		df_falliti = df_finale[df_finale['Status'] == 'Fallito']
		# Mostra il nome del dataset e il motivo dell'errore
		print(df_falliti[['Dataset', 'Note_Errore']].to_string(index=False))

Inizio analisi GDV su 47 dataset...

Elaborazione: Autism Screening Adult (ID: 426)... Completato! GDV: -0.0384 | Tempo: 1.54s
Elaborazione: Bank Marketing (ID: 222)... Completato! GDV: -0.0215 | Tempo: 51.98s
Elaborazione: Blood Transfusion (ID: 176)... Completato! GDV: -0.0386 | Tempo: 1.81s
Elaborazione: Breast Cancer (ID: 14)... Completato! GDV: -0.0084 | Tempo: 1.54s
Elaborazione: Breast Cancer Coimbra (ID: 451)... Completato! GDV: -0.0235 | Tempo: 1.09s
Elaborazione: Breast Cancer Wisconsin (Diag.) (ID: 17)... Completato! GDV: -0.2142 | Tempo: 1.59s
Elaborazione: Cardiotocography (ID: 193)... FALLITO! Motivo: IndexError: The shape of the mask [4252] at index 0 does not match the shape of the indexed tensor [2126, 2126] at index 0
Elaborazione: Chess (King-Rook vs. Pawn) (ID: 22)... FALLITO! Motivo: RuntimeError: [enforce fail at alloc_cpu.cpp:114] data. DefaultCPUAllocator: not enough memory: you tried to allocate 268132110067600 bytes.
Elaborazione: Cirrhosis Patient Survival (I

In [ ]:
# ==========================================
# CONFIGURAZIONE DINAMICA DEL DATASET
# ==========================================
# Modifica queste variabili quando passi a un nuovo dataset
DATASET_NAME = "Letter Recognition"
DATASET_ID = 59 

# Creiamo una versione "sicura" del nome da usare nei file (senza spazi)
DATASET_NAME_SAFE = DATASET_NAME.replace(" ", "_").replace("/", "-")

print(f"=== INIZIALIZZAZIONE PIPELINE: {DATASET_NAME} ===")

# Per valutare il GDV di partenza, scaliamo l'intero dataset. 
# NOTA: Questo scaling serve SOLO per la valutazione geometrica iniziale, non per il training!
X_scaled_baseline = StandardScaler().fit_transform(X_raw)

X_tensor = torch.tensor(X_scaled_baseline, dtype=torch.float32)
y_tensor = torch.tensor(y_raw, dtype=torch.long)

GDV_INIZIALE = calculate_gdv(X_tensor, y_tensor)

print(f"Dataset ID: {DATASET_ID}")
print(f"GDV Dati Grezzi: {GDV_INIZIALE:.4f}")
print("===================================================")

In [ ]:
# === ESPERIMENTO 1: BASELINE (CROSS-ENTROPY) ===
print(f"\nAvvio Addestramento: Cross-Entropy su {DATASET_NAME} (Hold-Out)")

torch.manual_seed(42)
model_ce = MLP(input_size=16, num_classes=num_classes).to(device)
criterion_ce = nn.CrossEntropyLoss()
optimizer_ce = optim.Adam(model_ce.parameters(), lr=0.001)

history_ce, best_epoch_ce = train_model(
	model=model_ce, criterion=criterion_ce, optimizer=optimizer_ce, 
	train_loader=train_loader, val_loader=val_loader, device=device,
	num_epochs=30, layer_for_gdv=model_ce.relu2
)

test_acc_ce = evaluate_model(model_ce, test_loader, device)
history_ce['test_acc'] = test_acc_ce
print(f"Accuratezza Test Finale: {test_acc_ce:.2f}%")

# MLOps: Salvataggio log in JSON e generazione Grafici PDF
utils.salva_storico_json(history_ce, f"HoldOut_CE_{DATASET_NAME_SAFE}", directory="logs")
plot_training_history(history_ce, f"Cross-Entropy ({DATASET_NAME})", GDV_INIZIALE, save_dir="plots")

In [ ]:
# === ESPERIMENTO 1: BASELINE (POLYLOSS) ===
print(f"\nAvvio Addestramento: PolyLoss su {DATASET_NAME} (Hold-Out)")

torch.manual_seed(42)
model_combinedLoss = MLP(input_size=16, num_classes=num_classes).to(device)
criterion_combinedLoss = PolyLoss()
optimizer_combinedLoss = optim.Adam(model_combinedLoss.parameters(), lr=0.001)

history_poly, best_epoch_poly = train_model(
	model=model_combinedLoss, 
	criterion=criterion_combinedLoss, 
	optimizer=optimizer_combinedLoss, 
	train_loader=train_loader, 
	val_loader=val_loader, 
	device=device,
	num_epochs=30, 
	layer_for_gdv=model_combinedLoss.relu2
)

test_acc_poly = evaluate_model(model_combinedLoss, test_loader, device)
history_poly['test_acc'] = test_acc_poly
print(f"Accuratezza Test Finale: {test_acc_poly:.2f}%")

# MLOps: Salvataggio log in JSON e generazione Grafici PDF
utils.salva_storico_json(history_poly, f"HoldOut_PolyLoss_{DATASET_NAME_SAFE}", directory="logs")
plot_training_history(history_poly, f"PolyLoss ({DATASET_NAME})", GDV_INIZIALE, save_dir="plots")

In [ ]:
# === ESPERIMENTO 1: BASELINE (C-Loss) ===
print(f"\nAvvio Addestramento: Combined C-Loss su {DATASET_NAME} (Hold-Out)")

torch.manual_seed(42)
model_combinedLoss = MLP(input_size=16, num_classes=num_classes).to(device)
criterion_combinedLoss = CombinedCLoss()
optimizer_combinedLoss = optim.Adam(model_combinedLoss.parameters(), lr=0.001)

history_poly, best_epoch_poly = train_model(
	model=model_combinedLoss, 
	criterion=criterion_combinedLoss, 
	optimizer=optimizer_combinedLoss, 
	train_loader=train_loader, 
	val_loader=val_loader, 
	device=device,
	num_epochs=30, 
	layer_for_gdv=model_combinedLoss.relu2
)

test_acc_poly = evaluate_model(model_combinedLoss, test_loader, device)
history_poly['test_acc'] = test_acc_poly
print(f"Accuratezza Test Finale: {test_acc_poly:.2f}%")

# MLOps: Salvataggio log in JSON e generazione Grafici PDF
utils.salva_storico_json(history_poly, f"HoldOut_CombinedLoss_{DATASET_NAME_SAFE}", directory="logs")
plot_training_history(history_poly, f"Combined C-Loss ({DATASET_NAME})", GDV_INIZIALE, save_dir="plots")

In [ ]:
import numpy as np
import torch
import random
import os
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

# --- FUNZIONE SCUDO PER LA RIPRODUCIBILITÀ ---
def imposta_seed(seed=42):
	"""Fissa tutti i generatori di numeri casuali per garantire riproducibilità."""
	random.seed(seed)
	os.environ['PYTHONHASHSEED'] = str(seed)
	np.random.seed(seed)
	torch.manual_seed(seed)
	if torch.cuda.is_available():
		torch.cuda.manual_seed(seed)
		torch.cuda.manual_seed_all(seed)
		torch.backends.cudnn.deterministic = True
		torch.backends.cudnn.benchmark = False

# === ESPERIMENTO K-FOLD CON GDV E SEED ROLLING ===
def run_kfold_experiment(loss_fn, model_name, X, y, k=10, epochs=30):
	print(f"\n=== AVVIO {k}-FOLD CV: {model_name} ===")
	
	# Fissiamo il seed globale per garantire che la divisione in fold sia identica per ogni Loss
	imposta_seed(42)
	skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
	
	fold_accs = []
	fold_gdvs = []
	
	for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
		print(f"\n--- Fold {fold+1}/{k} ---")
		
		# --- SEED ROLLING: La garanzia dell'equità strutturale ---
		# Il Fold N partirà sempre dalla stessa configurazione sinaptica, per qualsiasi Loss
		seed_corrente = 42 + fold
		imposta_seed(seed_corrente)
		# ---------------------------------------------------------
		
		# 1. Split corretto
		X_tr, X_va = X[train_idx], X[val_idx]
		y_tr, y_va = y[train_idx], y[val_idx]
		
		# 2. Scaling corretto (No Data Leakage!)
		scaler = StandardScaler()
		X_tr_scaled = scaler.fit_transform(X_tr)
		X_va_scaled = scaler.transform(X_va)
		
		# 3. Dataloaders
		tr_loader = create_dataloader(X_tr_scaled, y_tr, batch_size=64, shuffle=True)
		va_loader = create_dataloader(X_va_scaled, y_va, batch_size=64, shuffle=False)
		
		# 4. Modello pulito (Inizializzato ORA con il seed_corrente)
		model = MLP(input_size=16, num_classes=num_classes).to(device)
		optimizer = optim.Adam(model.parameters(), lr=0.001)
		
		# 5. Addestramento
		history, best_ep = train_model(
			model=model, criterion=loss_fn, optimizer=optimizer,
			train_loader=tr_loader, val_loader=va_loader, device=device,
			num_epochs=epochs, layer_for_gdv=model.relu2 # Assicurati che model.relu2 sia corretto per la tua architettura
		)
		
		# 6. Registriamo i risultati (dal modello ripristinato al Best Checkpoint)
		acc = evaluate_model(model, va_loader, device)
		fold_accs.append(acc)
		
		# Prendiamo il GDV corrispondente all'epoca migliore salvata
		best_gdv = history['val_gdv'][best_ep - 1] if history['val_gdv'] else 0.0
		fold_gdvs.append(best_gdv)
		
	return fold_accs, fold_gdvs

# ... [Assumendo che la funzione run_kfold_experiment sia già definita come l'avevamo scritta] ...

print(f"\n=== ESECUZIONE 10-FOLD CV SU {DATASET_NAME} ===")

# Esecuzione per tutti e 3 i modelli
acc_ce, gdv_ce = run_kfold_experiment(nn.CrossEntropyLoss(), "Cross-Entropy", X_raw, y_raw)
acc_poly, gdv_poly = run_kfold_experiment(PolyLoss(epsilon=2.0), "PolyLoss", X_raw, y_raw)
acc_closs, gdv_closs = run_kfold_experiment(CombinedCLoss(sigma=0.5, gamma=0.5), "Combined C-Loss", X_raw, y_raw)

# MLOps: Salviamo i risultati grezzi del K-Fold in JSON!
kfold_results = {
	"dataset": DATASET_NAME,
	"folds": 10,
	"Cross-Entropy": {"acc": acc_ce, "gdv": gdv_ce},
	"PolyLoss": {"acc": acc_poly, "gdv": gdv_poly},
	"Combined C-Loss": {"acc": acc_closs, "gdv": gdv_closs}
}
utils.salva_storico_json(kfold_results, f"KFold_Results_{DATASET_NAME_SAFE}", directory="logs")

print("\n" + "="*50)
print("--- RIEPILOGO FINALE K-FOLD (Accuratezza Media) ---")
print(f"1. Cross-Entropy: {np.mean(acc_ce):.2f}% (± {np.std(acc_ce):.2f}%)")
print(f"2. PolyLoss:	  {np.mean(acc_poly):.2f}% (± {np.std(acc_poly):.2f}%)")
print(f"3. C-Loss:		{np.mean(acc_closs):.2f}% (± {np.std(acc_closs):.2f}%)")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
from datetime import datetime

# ==========================================
# CONFIGURAZIONE GLOBALE PER STAMPA TESI 
# ==========================================
plt.rcParams.update({
	'font.family': 'serif',
	'font.size': 11,
	'axes.titlesize': 13,
	'axes.labelsize': 11,
	'xtick.labelsize': 10,
	'ytick.labelsize': 10,
	'legend.fontsize': 10,
	'lines.linewidth': 1.0,
	'figure.dpi': 300,
	'grid.linewidth': 0.5,
	'grid.alpha': 0.5
})

# ==========================================
# ESECUZIONE GRAFICO 10-FOLD
# ==========================================
fig, ax1 = plt.subplots(figsize=(5.5, 4.0))

modelli = ['Cross-Entropy', 'PolyLoss', 'Combined C-Loss']
medie_acc = [np.mean(acc_ce), np.mean(acc_poly), np.mean(acc_closs)]
std_acc = [np.std(acc_ce), np.std(acc_poly), np.std(acc_closs)]

# Colori ad alto contrasto per la stampa
colori = ['#004488', '#BB5500', '#117733']

# Spessori dei bordi e delle error bars impostati a 1.0
barre = ax1.bar(modelli, medie_acc, yerr=std_acc, capsize=5, 
				color=colori, alpha=0.85, edgecolor='black', linewidth=1.0,
				error_kw={'elinewidth': 1.0, 'capthick': 1.0})

# Adatta lo zoom in base ai dati reali lasciando margine
min_acc = min(medie_acc) - max(std_acc) - 1.5
max_acc = max(medie_acc) + max(std_acc) + 1.5
ax1.set_ylim(min_acc, max_acc)

ax1.set_ylabel('Accuratezza Media (%)')
ax1.set_title(f'Confronto Prestazioni 10-Fold Cross Validation\n(Dataset: {DATASET_NAME})')

# Inserimento dinamico delle percentuali all'interno delle barre
for i, (barra, media) in enumerate(zip(barre, medie_acc)):
	altezza = barra.get_height()
	base_error_bar = altezza - std_acc[i]
	y_text = min_acc + (base_error_bar - min_acc) / 2
	
	ax1.text(barra.get_x() + barra.get_width()/2., y_text, 
			 f'{media:.2f}%', ha='center', va='center', 
			 color='white', fontweight='bold', fontsize=10)

plt.grid(axis='y', linestyle='--')
plt.tight_layout()

# ==========================================
# SALVATAGGIO AUTOMATICO (MLOps)
# ==========================================
save_dir = "plots"
os.makedirs(save_dir, exist_ok=True)
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
filename_pdf = os.path.join(save_dir, f"{timestamp}_KFold_{DATASET_NAME_SAFE}_Acc.pdf")

plt.savefig(filename_pdf, format='pdf', bbox_inches='tight')
print(f"[*] Grafico K-Fold salvato per la tesi: {filename_pdf}")

plt.show()